# 03. 형상과 Mask 자동 생성

돌출된 메탈 형상과 pixel-level segmentation mask를 자동 생성합니다.

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "synthetic_metal_utils.py").exists():
    search_roots = [Path.cwd(), *Path.cwd().parents]
    search_patterns = ["synthetic_metal_utils.py", "*/synthetic_metal_utils.py", "*/*/synthetic_metal_utils.py"]
    for root in search_roots:
        for pattern in search_patterns:
            matches = list(root.glob(pattern))
            if matches:
                NOTEBOOK_DIR = matches[0].parent
                break
        if (NOTEBOOK_DIR / "synthetic_metal_utils.py").exists():
            break

sys.path.append(str(NOTEBOOK_DIR))
DATA_ROOT = NOTEBOOK_DIR / "data" / "synthetic_metal_seg"

from synthetic_metal_utils import *
set_korean_font()
set_seed(7)

print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("DATA_ROOT:", DATA_ROOT)

## 형상 sample 생성

같은 class라도 위치, 크기, 회전, shape type이 달라지도록 만듭니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
fig, axes = plt.subplots(2, 6, figsize=(10, 3.5))
for idx in range(6):
    mask, meta = generate_shape_mask(rng, 128)
    axes[0, idx].imshow(mask, cmap="gray")
    axes[0, idx].set_title(meta["shape_type"], fontsize=8)
    axes[0, idx].axis("off")
    axes[1, idx].text(0.02, 0.5, str(meta), fontsize=7, wrap=True)
    axes[1, idx].axis("off")
plt.tight_layout()

## Mask 품질 기준

mask는 자동 생성되므로 정확해야 합니다. 이후 성능 하락은 label noise가 아니라 domain shift 때문이어야 합니다.

In [ ]:
mask_values = sorted(np.unique(mask).tolist())
mask_values